In [1]:
import os
import pickle
import numpy as np
from collections import Counter

# Path to the NTU 120 one-shot annotation file
# PKL_PATH = "ntu60_2d.pkl"
PKL_PATH = "ntu120_2d.pkl"
# PKL_PATH = "../../pyskl/data/ucf101/ucf101_hrnet.pkl"

with open(PKL_PATH, "rb") as f:
    obj = pickle.load(f)

print("Loaded:", PKL_PATH)
print("Top-level keys:", obj.keys())

Loaded: ntu120_2d.pkl
Top-level keys: dict_keys(['split', 'annotations'])


In [2]:
split = obj["split"]
anns = obj["annotations"]

print("Split keys:", split.keys())
print("Split sizes:", {k: len(v) for k, v in split.items()})
print("Num annotations:", len(anns))

# Show a few sequence IDs from each split
for k in split:
    print(f"\n[{k}] first 5 ids:")
    print(split[k][:5])

Split keys: dict_keys(['train', 'exemplar', 'test'])
Split sizes: {'train': 95001, 'exemplar': 20, 'test': 18924}
Num annotations: 113945

[train] first 5 ids:
['S001C001P001R001A002', 'S001C001P001R001A003', 'S001C001P001R001A004', 'S001C001P001R001A005', 'S001C001P001R001A006']

[exemplar] first 5 ids:
['S001C003P008R001A001', 'S001C003P008R001A007', 'S001C003P008R001A013', 'S001C003P008R001A019', 'S001C003P008R001A025']

[test] first 5 ids:
['S001C001P001R001A001', 'S001C001P001R001A007', 'S001C001P001R001A013', 'S001C001P001R001A019', 'S001C001P001R001A025']


In [3]:
sample = anns[0]

print("Annotation keys:", sample.keys())
print("frame_dir:", sample["frame_dir"])
print("label:", sample["label"])
print("total_frames:", sample["total_frames"])

kp = sample["keypoint"]
ks = sample["keypoint_score"]

print("\nkeypoint: type =", type(kp), "dtype =", kp.dtype, "shape =", kp.shape)
print("keypoint_score: type =", type(ks), "dtype =", ks.dtype, "shape =", ks.shape)

Annotation keys: dict_keys(['start_frame', 'end_frame', 'pos_label', 'frame_dir', 'img_shape', 'original_shape', 'total_frames', 'keypoint', 'keypoint_score', 'source', 'label'])
frame_dir: S001C001P001R001A001
label: 1
total_frames: 103

keypoint: type = <class 'numpy.ndarray'> dtype = float16 shape = (1, 103, 25, 2)
keypoint_score: type = <class 'numpy.ndarray'> dtype = float16 shape = (1, 103, 25, 1)


In [4]:
import random

# Randomly inspect several samples to verify consistency
idxs = random.sample(range(len(anns)), 5)

for i in idxs:
    s = anns[i]
    kp = s["keypoint"]
    ks = s["keypoint_score"]
    print(f"\n[{i}] frame_dir={s['frame_dir']} label={s['label']} total_frames={s['total_frames']}")
    print("  keypoint    -> dtype:", kp.dtype, "shape:", kp.shape)
    print("  keypt_score -> dtype:", ks.dtype, "shape:", ks.shape)


[24353] frame_dir=S008C002P032R002A044 label=44 total_frames=61
  keypoint    -> dtype: float16 shape: (1, 61, 25, 2)
  keypt_score -> dtype: float16 shape: (1, 61, 25, 1)

[24715] frame_dir=S008C002P035R002A051 label=51 total_frames=47
  keypoint    -> dtype: float16 shape: (1, 47, 25, 2)
  keypt_score -> dtype: float16 shape: (1, 47, 25, 1)

[35825] frame_dir=S011C003P002R001A024 label=24 total_frames=59
  keypoint    -> dtype: float16 shape: (1, 59, 25, 2)
  keypt_score -> dtype: float16 shape: (1, 59, 25, 1)

[21119] frame_dir=S007C003P025R002A024 label=24 total_frames=73
  keypoint    -> dtype: float16 shape: (1, 73, 25, 2)
  keypt_score -> dtype: float16 shape: (1, 73, 25, 1)

[78384] frame_dir=S025C003P055R001A098 label=98 total_frames=60
  keypoint    -> dtype: float16 shape: (1, 60, 25, 2)
  keypt_score -> dtype: float16 shape: (1, 60, 25, 1)


In [5]:
# Build a mapping from frame_dir → index in annotations
frame_to_idx = {}
for i, s in enumerate(anns):
    fd = s["frame_dir"]
    if fd in frame_to_idx:
        print("Warning: duplicate frame_dir in annotations:", fd)
    frame_to_idx[fd] = i

print("Unique frame_dir count:", len(frame_to_idx))

# Verify that all IDs in each split exist in annotations
for k in split:
    missing = [fid for fid in split[k] if fid not in frame_to_idx]
    print(f"{k}: {len(split[k])} ids, missing in annotations: {len(missing)}")
    if missing:
        print("  example missing ids:", missing[:5])

# Check that train / exemplar / test are disjoint
train_set = set(split["train"])
ex_set = set(split["exemplar"])
test_set = set(split["test"])

print("\nIntersections:")
print("train ∩ exemplar:", len(train_set & ex_set))
print("train ∩ test:", len(train_set & test_set))
print("exemplar ∩ test:", len(ex_set & test_set))

Unique frame_dir count: 113945
train: 95001 ids, missing in annotations: 0
exemplar: 20 ids, missing in annotations: 0
test: 18924 ids, missing in annotations: 0

Intersections:
train ∩ exemplar: 0
train ∩ test: 0
exemplar ∩ test: 0


In [6]:
# The 20 novel action labels in NTU RGB+D 120 one-shot protocol
novel_classes = {
    1, 7, 13, 19, 25,
    31, 37, 43, 49, 55,
    61, 67, 73, 79, 85,
    91, 97, 103, 109, 115,
}

# Collect labels for exemplar and test splits
exemplar_labels = []
test_labels = []

for fid in split["exemplar"]:
    s = anns[frame_to_idx[fid]]
    exemplar_labels.append(s["label"])

for fid in split["test"]:
    s = anns[frame_to_idx[fid]]
    test_labels.append(s["label"])

print("Exemplar labels count:", Counter(exemplar_labels))
print("\nNum novel classes with exemplar:", len(set(exemplar_labels)))
print("Novel label set in exemplar:", sorted(set(exemplar_labels)))

print("\nTest labels (novel classes) summary:")
c_test = Counter(test_labels)
print("Num novel labels in test:", len(c_test))
print("Sample counts per novel class (test only):")
for lbl in sorted(c_test.keys()):
    print(f"  label {lbl}: {c_test[lbl]} samples")

Exemplar labels count: Counter({1: 1, 7: 1, 13: 1, 19: 1, 25: 1, 31: 1, 37: 1, 43: 1, 49: 1, 55: 1, 61: 1, 67: 1, 73: 1, 79: 1, 85: 1, 91: 1, 97: 1, 103: 1, 109: 1, 115: 1})

Num novel classes with exemplar: 20
Novel label set in exemplar: [1, 7, 13, 19, 25, 31, 37, 43, 49, 55, 61, 67, 73, 79, 85, 91, 97, 103, 109, 115]

Test labels (novel classes) summary:
Num novel labels in test: 20
Sample counts per novel class (test only):
  label 1: 939 samples
  label 7: 943 samples
  label 13: 937 samples
  label 19: 941 samples
  label 25: 945 samples
  label 31: 943 samples
  label 37: 945 samples
  label 43: 945 samples
  label 49: 945 samples
  label 55: 905 samples
  label 61: 934 samples
  label 67: 953 samples
  label 73: 953 samples
  label 79: 955 samples
  label 85: 956 samples
  label 91: 955 samples
  label 97: 957 samples
  label 103: 957 samples
  label 109: 958 samples
  label 115: 958 samples


In [ ]:
all_labels = [s["label"] for s in anns]
label_counts = Counter(all_labels)

print("Total distinct labels:", len(label_counts))
print("Label range:", min(label_counts.keys()), "→", max(label_counts.keys()))

print("\nSample count for first 20 labels:")
for lbl in sorted(label_counts.keys())[:20]:
    print(f"  label {lbl}: {label_counts[lbl]} samples")

Total distinct labels: 120
Label range: 1 → 120

Sample count for first 20 labels:
  label 1: 940 samples
  label 2: 941 samples
  label 3: 938 samples
  label 4: 940 samples
  label 5: 942 samples
  label 6: 943 samples
  label 7: 944 samples
  label 8: 941 samples
  label 9: 936 samples
  label 10: 941 samples
  label 11: 937 samples
  label 12: 932 samples
  label 13: 938 samples
  label 14: 943 samples
  label 15: 945 samples
  label 16: 940 samples
  label 17: 943 samples
  label 18: 941 samples
  label 19: 942 samples
  label 20: 939 samples


: 